# Lecture 8 — Class Exercise
## Choropleth Maps

> **Push to:** `week08/lecture08_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json
import urllib.request


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [ ]:
# ── Task 1: World choropleth — life expectancy vs global average ─────────────

# Step 1: Load Gapminder and filter to 2007
gm      = px.data.gapminder()
gm_2007 = gm.loc[gm['year'] == 2007].copy()

# Step 2: Compute deviation from the global mean
global_avg           = gm_2007['lifeExp'].mean()
gm_2007['lifeExp_vs_avg'] = gm_2007['lifeExp'] - global_avg

print(f"Global average life expectancy (2007): {global_avg:.1f} years")
print(f"Range of deviation: {gm_2007['lifeExp_vs_avg'].min():.1f} to {gm_2007['lifeExp_vs_avg'].max():.1f} years")

# Step 3: Build the choropleth
# CHART CHOICE: px.choropleth — static projection is best here.
#   This is a world overview meant to show regional patterns, not an
#   interactive drilldown. A static projection renders cleanly in
#   presentations and reports.
#
# COLOUR SCALE CHOICE: diverging (RdBu).
#   The data is above/below a meaningful midpoint (world average = 0),
#   not a one-directional quantity. A diverging scale makes it
#   immediately obvious which countries are below average (red)
#   vs above average (blue), with white = exactly at the world average.

fig1 = px.choropleth(
    gm_2007,
    locations='iso_alpha',
    locationmode='ISO-3',
    color='lifeExp_vs_avg',
    color_continuous_scale='RdBu',          # diverging: red = below avg, blue = above
    color_continuous_midpoint=0,            # white at zero = world average
    hover_name='country',
    hover_data={
        'lifeExp':       ':.1f',            # raw life expectancy in years
        'lifeExp_vs_avg':':.1f',            # deviation from world average
        'iso_alpha':     False,             # hide the iso code from hover
    },
    labels={
        'lifeExp':        'Life Expectancy (yrs)',
        'lifeExp_vs_avg': f'vs World Avg ({global_avg:.1f} yrs)',
    },
    # Insight title: geographic finding — names sub-Saharan Africa as the outlier
    title=f'Sub-Saharan Africa falls up to 30 years below the world average — no other region comes close (world avg: {global_avg:.1f} yrs)',
    projection='natural earth',
    height=500,
)

# Step 4: Layout polish — professor's style
fig1.update_layout(
    font=dict(family='Arial', size=11),
    coloraxis_colorbar=dict(
        title=f'vs World Avg<br>({global_avg:.1f} yrs)',
        thickness=15,
        len=0.6
    ),
    geo=dict(showframe=False),
    margin=dict(l=0, r=0, t=60, b=0),
)

fig1.show()


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** Germany federal states boundaries — `https://raw.githubusercontent.com/isellsoap/deutschlandGeoJSON/main/2_bundeslaender/4_niedrig.geo.json` (public domain, same source used in the lecture notes)

**Dataset:** Unemployment rate (%) by German federal state, 2023 (Bundesagentur für Arbeit)

**Chart type chosen:** `px.choropleth` — the goal is a clean, static overview of regional unemployment inequality across Germany's 16 states, not an interactive map for exploration. A static projection is appropriate for presentations and reports and matches the lecture examples for sub-country maps.

**Colour scale chosen:** **Sequential (`Oranges`)** — unemployment rate is a one-directional quantity (there is no meaningful "below average" story here; even the lowest rate is still a positive value). A sequential scale from light (low unemployment) to dark (high unemployment) correctly encodes the magnitude. A diverging scale would only be justified if we were plotting *deviation from average*, like Task 1.


In [ ]:
# ── Task 2: Germany federal states — unemployment rate 2023 ──────────────────

# Step 1: Load GeoJSON directly from GitHub (same file used in lecture notes)
geojson_url = (
    'https://raw.githubusercontent.com/isellsoap/deutschlandGeoJSON'
    '/main/2_bundeslaender/4_niedrig.geo.json'
)
with urllib.request.urlopen(geojson_url) as response:
    germany_geojson = json.load(response)

# Step 2: Inspect the properties to find the right featureidkey
# This is the professor's mandatory first step before any custom GeoJSON chart
print("GeoJSON properties:", germany_geojson['features'][0]['properties'])
# → will show 'name' as the state name key → featureidkey='properties.name'


In [ ]:
# Step 3: Unemployment data per German federal state (2023, Bundesagentur für Arbeit)
unemployment = pd.DataFrame({
    'State': [
        'Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen',
        'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen',
        'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland',
        'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen'
    ],
    'Unemployment_Rate': [
        3.7, 2.8, 9.2, 6.5, 10.2,
        7.0, 4.8, 8.4, 5.5,
        7.4, 4.3, 6.4,
        6.1, 8.8, 5.7, 6.7
    ]
})

print(unemployment.sort_values('Unemployment_Rate', ascending=False))

# Step 4: Build the choropleth
fig2 = px.choropleth(
    data_frame=unemployment,
    geojson=germany_geojson,
    locations='State',                      # column in dataframe
    featureidkey='properties.name',         # matched from the properties inspection above
    color='Unemployment_Rate',
    color_continuous_scale='Oranges',       # sequential — one-directional quantity
    range_color=[2, 11],                    # anchored to actual data range
    fitbounds='locations',                  # auto-zoom to Germany
    hover_name='State',
    hover_data={'State': False, 'Unemployment_Rate': ':.1f'},
    labels={'Unemployment_Rate': 'Unemployment Rate (%)'},
    # Insight title: geographic finding — names the east-west divide
    title='The east-west divide persists — Bremen and Berlin top 9%, while Bayern stays below 3%',
    height=600,
)

# Step 5: Layout polish
fig2.update_layout(
    font=dict(family='Arial', size=12),
    geo=dict(showframe=False, showcoastlines=False),
    coloraxis_colorbar=dict(title='Unemployment<br>Rate (%)', thickness=15, len=0.6),
    margin=dict(l=0, r=0, t=60, b=0),
)

fig2.show()
